In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data/nifty100.db")

pd.read_sql("""
SELECT COUNT(*) AS company_count
FROM companies;
""", conn)

,company_count
0,92


In [5]:
import pandas as pd

query = """
SELECT
    c.id,
    c.company_name,

    COUNT(DISTINCT pl.year) AS pl_years,
    COUNT(DISTINCT bs.year) AS bs_years,
    COUNT(DISTINCT cf.year) AS cf_years

FROM companies c

LEFT JOIN profitandloss pl
ON c.id = pl.company_id

LEFT JOIN balancesheet bs
ON c.id = bs.company_id

LEFT JOIN cashflow cf
ON c.id = cf.company_id

GROUP BY c.id, c.company_name;
"""

df = pd.read_sql(query, conn)

qualified = df[
    (df["pl_years"] >= 10) &
    (df["bs_years"] >= 10) &
    (df["cf_years"] >= 10)
]

percentage = (len(qualified) / len(df)) * 100

print(f"Qualified Companies : {len(qualified)}")
print(f"Total Companies     : {len(df)}")
print(f"Percentage          : {percentage:.2f}%")

Qualified Companies : 84
Total Companies     : 92
Percentage          : 91.30%


In [38]:
import pandas as pd

fk_check = pd.read_sql("PRAGMA foreign_key_check;", conn)

print(fk_check)

if fk_check.empty:
    print("\n✅ PASS - No foreign key violations found.")
else:
    print(f"\n❌ FAIL - {len(fk_check)} foreign key violations found.")

Empty DataFrame
Columns: [table, rowid, parent, fkid]
Index: []

✅ PASS - No foreign key violations found.


In [39]:
import pandas as pd

result = pd.read_sql("""
SELECT COUNT(*) AS total_ratios
FROM financial_ratios;
""", conn)

count = result.loc[0, "total_ratios"]

print(result)

if count >= 1100:
    print(f"\n✅ PASS - {count} records found.")
else:
    print(f"\n❌ FAIL - Only {count} records found.")

   total_ratios
0          1160

✅ PASS - 1160 records found.


In [40]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("data/nifty100.db")

# Pick one company with at least 6 years of sales data
company = "ABB"   # We can use ABB for the spot check

sales = pd.read_sql(f"""
SELECT year, sales
FROM profitandloss
WHERE company_id = '{company}'
ORDER BY year;
""", conn)

print(sales)

        year   sales
0   Dec 2012  1653.0
1   Mar 2014  2276.0
2   Mar 2015  2289.0
3   Mar 2016  2614.0
4   Mar 2017  2903.0
5   Mar 2018  3298.0
6   Mar 2019  3679.0
7   Mar 2020  4093.0
8   Mar 2021  4310.0
9   Mar 2022  4913.0
10  Mar 2023  5349.0
11  Mar 2024  5849.0
12       TTM  6066.0


In [41]:
import pandas as pd

# Manual 5-Year Revenue CAGR
start_sales = 3679
end_sales = 5849
years = 5

manual_cagr = ((end_sales / start_sales) ** (1 / years) - 1) * 100

print(f"Manual Revenue CAGR: {manual_cagr:.2f}%")

# Stored value
stored = pd.read_sql("""
SELECT year, revenue_cagr_5yr
FROM financial_ratios
WHERE company_id = 'ABB'
ORDER BY year DESC
LIMIT 5;
""", conn)

print(stored)

Manual Revenue CAGR: 9.72%
       year  revenue_cagr_5yr
0  Mar 2024              9.72
1  Mar 2024              9.72
2  Mar 2023             10.16
3  Mar 2023             10.16
4  Mar 2022             11.10


In [49]:
import pandas as pd

query = """
SELECT
    c.id,
    c.company_name,
    c.roe_percentage AS company_roe,
    fr.return_on_equity_pct AS calculated_roe,
    ABS(c.roe_percentage - fr.return_on_equity_pct) AS difference
FROM companies c
JOIN financial_ratios fr
    ON c.id = fr.company_id
WHERE fr.year = 'Mar 2024'
LIMIT 5;
"""

df = pd.read_sql(query, conn)

print(df)

if (df["difference"] <= 5).all():
    print("\n✅ PASS - All 5 companies are within ±5%.")
else:
    print("\n❌ FAIL - One or more companies differ by more than 5%.")

           id                company_name  company_roe  calculated_roe  \
0         ABB            Abbott India Ltd        34.90           32.47   
1         ABB            Abbott India Ltd        34.90           32.47   
2  ADANIENSOL  Adani Energy Solutions Ltd         8.59            9.46   
3    ADANIENT       Adani Enterprises Ltd         8.53            8.53   
4  ADANIGREEN      Adani Green Energy Ltd        14.70           12.81   

   difference  
0        2.43  
1        2.43  
2        0.87  
3        0.00  
4        1.89  

✅ PASS - All 5 companies are within ±5%.


In [55]:
import pandas as pd

df = pd.read_excel("output/screener_output.xlsx")  # adjust path if needed

print(df["year"].value_counts())

year
Mar 2024    23
Mar 2023    21
Mar 2022    18
Mar 2019    17
Mar 2018    14
Mar 2020    12
Mar 2021     8
Dec 2020     1
Dec 2023     1
Dec 2022     1
Dec 2021     1
Sep 2024     1
Name: count, dtype: int64


In [56]:
import pandas as pd

df = pd.read_excel("output/screener_output.xlsx")

latest = df[df["year"] == "Mar 2024"]

print("Latest rows:", len(latest))

latest[[
    "company_id",
    "return_on_equity_pct",
    "debt_to_equity",
    "revenue_cagr_5yr",
    "composite_quality_score"
]].head()

Latest rows: 23


,company_id,return_on_equity_pct,debt_to_equity,revenue_cagr_5yr,composite_quality_score
0,INDIGO,892.57,0.0186,19.31,87.08
4,TCS,50.94,0.0886,10.46,8.98
13,TRENT,36.31,0.4309,36.31,59.65
14,IRCTC,34.40,0.0186,17.96,25.20
15,NESTLEIND,117.75,0.1033,14.55,14.95


In [58]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table';
""", conn)

,name
0,companies
1,balancesheet
2,profitandloss
3,cashflow
4,financial_ratios
5,market_cap
6,peer_groups
7,sectors
8,stock_prices
9,analysis


In [59]:
import pandas as pd

pd.read_sql("""
SELECT *
FROM peer_groups
LIMIT 10;
""", conn)

,id,peer_group_name,company_id,is_benchmark
0,1,Private Banks,HDFCBANK,1
1,2,Private Banks,ICICIBANK,0
2,3,Private Banks,AXISBANK,0
3,4,Private Banks,KOTAKBANK,0
4,5,Private Banks,INDUSINDBK,0
5,6,Public Sector Banks,SBIN,1
6,7,Public Sector Banks,BANKBARODA,0
7,8,Public Sector Banks,CANBK,0
8,9,Public Sector Banks,PNB,0
9,10,IT Services,TCS,1


In [60]:
pd.read_sql("""
SELECT COUNT(DISTINCT peer_group)
FROM peer_groups;
""", conn)

DatabaseError: Execution failed on sql '
SELECT COUNT(DISTINCT peer_group)
FROM peer_groups;
': no such column: peer_group

In [61]:
pd.read_sql("""
PRAGMA table_info(peer_groups);
""", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,peer_group_name,TEXT,0,None,0
2,2,company_id,TEXT,0,None,0
3,3,is_benchmark,INTEGER,0,None,0


In [62]:
pd.read_sql("""
SELECT *
FROM peer_groups
LIMIT 10;
""", conn)

,id,peer_group_name,company_id,is_benchmark
0,1,Private Banks,HDFCBANK,1
1,2,Private Banks,ICICIBANK,0
2,3,Private Banks,AXISBANK,0
3,4,Private Banks,KOTAKBANK,0
4,5,Private Banks,INDUSINDBK,0
5,6,Public Sector Banks,SBIN,1
6,7,Public Sector Banks,BANKBARODA,0
7,8,Public Sector Banks,CANBK,0
8,9,Public Sector Banks,PNB,0
9,10,IT Services,TCS,1


In [63]:
pd.read_sql("""
SELECT
    COUNT(DISTINCT peer_group_name) AS total_peer_groups
FROM peer_groups;
""", conn)

,total_peer_groups
0,11


In [64]:
import pandas as pd

df = pd.read_csv("output/cluster_labels.csv")

print("Rows:", len(df))
print("Unique companies:", df["company_id"].nunique())
print("Missing cluster_id:", df["cluster_id"].isna().sum())

Rows: 92
Unique companies: 92
Missing cluster_id: 0


In [67]:
import pandas as pd

df = pd.read_csv("output/pros_cons_generated.csv")

summary = (
    df.groupby(["company_id", "type"])
      .size()
      .unstack(fill_value=0)
)

# Ensure both columns exist even if one is absent
for col in ["pro", "con"]:
    if col not in summary.columns:
        summary[col] = 0

failed = summary[(summary["pro"] < 1) | (summary["con"] < 1)]

print("Total companies:", len(summary))
print("Companies failing:", len(failed))

if len(failed):
    print(failed)

Total companies: 92
Companies failing: 0


In [70]:
from pathlib import Path

folder = Path("reports/tearsheets")

print(folder.exists())
print(folder.resolve())

False
C:\Users\Arush\OneDrive\Desktop\n100_financial_intelligence\reports\tearsheets


In [71]:
from pathlib import Path

pdfs = list(Path(".").rglob("*.pdf"))

print("Total PDFs found:", len(pdfs))

for pdf in pdfs[:20]:
    print(pdf)

Total PDFs found: 77
analyst_guide.pdf
reports\Abbott India Ltd.pdf
reports\Adani Energy Solutions Ltd.pdf
reports\Adani Enterprises Ltd.pdf
reports\Adani Green Energy Ltd.pdf
reports\Adani Power Ltd.pdf
reports\Adani Total Gas Ltd.pdf
reports\Ambuja Cements Ltd.pdf
reports\Avenue Supermarts Ltd.pdf
reports\Axis Bank Ltd.pdf
reports\Bajaj Auto Ltd.pdf
reports\Bajaj Finance Ltd.pdf
reports\Bajaj Finserv Ltd.pdf
reports\Bajaj Holdings & Investment Ltd.pdf
reports\Bank of Baroda.pdf
reports\Bharat Electronics Ltd.pdf
reports\Bharat Petroleum Corporation Ltd.pdf
reports\Bharti Airtel Ltd.pdf
reports\Bosch Ltd.pdf
reports\Britannia Industries Ltd.pdf


In [72]:
import pandas as pd

df = pd.read_csv("output/validation_failures.csv")

print(df.columns.tolist())
print(df.head())

['rule_id', 'severity', 'company_id', 'issue']
  rule_id severity  company_id                             issue
0   DQ-06  WARNING  ADANIENSOL  Sales less than or equal to zero
1   DQ-11  WARNING    ADANIENT            Invalid tax percentage
2   DQ-11  WARNING  ADANIGREEN            Invalid tax percentage
3   DQ-11  WARNING  ADANIGREEN            Invalid tax percentage
4   DQ-11  WARNING  ADANIGREEN            Invalid tax percentage


In [74]:
import requests
import time

url = "http://127.0.0.1:8000/api/v1/companies/TCS"

start = time.perf_counter()
r = requests.get(url)
end = time.perf_counter()

print("Status:", r.status_code)
print("Response Time:", round(end - start, 3), "seconds")

Status: 200
Response Time: 0.059 seconds


In [75]:
import pandas as pd

df = pd.read_csv("output/screener_output.csv")  # adjust filename if needed

print(df.shape)
print(df.columns.tolist())
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'output/screener_output.csv'